# RootIQ — AI Training Notebook (Colab Pro + Google Drive + Groq)

Everything here trains and **measures** models for RootIQ. Every stage saves to Google Drive (`MyDrive/RootIQ_AI`), so a Colab disconnect never loses work, and every stage ends with numbers you can put in front of judges.

| Stage | What it trains | Hardware | Why it helps RootIQ | Accuracy you get |
|---|---|---|---|---|
| **A** | Isolation-Forest anomaly model (the one the product already loads from `data/models/iforest.joblib`) | CPU, ~2 min | Catches *unusual combinations* of latency/loss/utilization; adds supporting evidence to RCA | TPR / FPR / detection delay / AUC vs the static thresholds |
| **B** | Training data for the language model: **vendor knowledge built from the repo** (43 vendors, 25 problems, 10 tasks, EN+AR, no API), RootIQ-grounded explanations, a networking tutor (Groq teacher), and optional licence-checked Hugging Face data | CPU (+ Groq) | Data you control, checked automatically: labels come from the same knowledge base the Vendor/Logs agents use | grounding pass rate; per-task row counts |
| **C** | QLoRA fine-tune of a small LLM (`Qwen3-4B-Instruct-2507` or Saudi `ALLaM-7B`) | GPU (T4 / L4 / A100) | An offline, private "network tutor + incident explainer + vendor advisor" that RootIQ can call instead of a cloud API | CCNA accuracy **before vs after**, grounding rate, **vendor-KB scores before vs after** |
| **D** | Evaluation report + ship/don't-ship decision | GPU | Stops you shipping a model that got worse or that invents/unsafe commands | `reports/eval_report.json` |
| **E** | Merge + save the model, and connect it to RootIQ (`LLM_PROVIDER=custom`) | GPU | The link between this notebook and the product | — |

**Rules of this notebook**
1. `SMOKE = True` runs tiny versions of every stage so you can check the pipeline in minutes. Switch to `False` only when the smoke run is clean.
2. Keys are read from **Colab Secrets** (the key icon in the left bar): add `GROQ_API_KEY` there. Never paste a key into a cell.
3. The RCA ranking stays deterministic code. The fine-tuned model is only a *tutor/explainer/advisor*; it never approves or executes anything.
4. The vendor data comes from the repo (`training/build_dataset.py`, branch `main` of your fork). Re-run cell 3 after changing the knowledge base; cell B0 tells you if the committed data is older than the knowledge base.
5. Public Hugging Face data is **off** (`INCLUDE_EXTERNAL = False`) and, when on, limited to datasets the catalog (`training/catalog/CATALOG.md`) approves for training.

In [ ]:
#@title 1) Setup: Drive, folders, secrets, switches (run first)
import os, sys, json, re, time, random, hashlib, pathlib, getpass, subprocess, threading, asyncio, datetime

SMOKE = True          # True = tiny run to verify the pipeline; False = the real run
INCLUDE_EXTERNAL = False  # True = also mix in licence-approved public Hugging Face data (see training/catalog/CATALOG.md); keep False for the first run
BASE_KEY = "qwen"     # "qwen" -> Qwen/Qwen3-4B-Instruct-2507 (fits a T4) | "allam" -> humain-ai/ALLaM-7B-Instruct-preview (Arabic/Saudi; use L4/A100)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = pathlib.Path("/content/drive/MyDrive/RootIQ_AI")
else:  # VS Code / local run
    ROOT = pathlib.Path(os.environ.get("ROOTIQ_AI_HOME", pathlib.Path.home() / "RootIQ_AI"))
for sub in ("data", "models", "reports", "cache", "checkpoints"):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
print("Workspace:", ROOT, "| SMOKE =", SMOKE)


def secret(name: str) -> str:
    """Colab Secrets first, then the environment, then a hidden prompt. The value is never printed or saved."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v:
                return v
        except Exception:
            pass
    return os.environ.get(name) or getpass.getpass(f"{name} (hidden, not saved): ")


def run_sync(coro):
    """Run an async function from a notebook cell (Colab already has a running event loop)."""
    box = {}

    def _t():
        try:
            box["r"] = asyncio.run(coro)
        except BaseException as e:  # noqa: BLE001
            box["e"] = e

    th = threading.Thread(target=_t)
    th.start()
    th.join()
    if "e" in box:
        raise box["e"]
    return box["r"]


def save_json(path, obj):
    pathlib.Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

In [ ]:
#@title 2) Install libraries (Colab only, ~2 min; if Colab asks to restart, do it once and rerun from cell 1)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "peft", "trl", "accelerate",
                    "bitsandbytes", "datasets", "scikit-learn", "joblib", "httpx", "pydantic-settings", "matplotlib"], check=True)
    print("Installed.")
else:
    print("Local run: install requirements yourself (see the README of the repo).")

In [ ]:
#@title 3) Get the RootIQ code (simulator, thresholds, templates, playbooks AND the vendor knowledge base + training data builders)
REPO_URL, BRANCH = "https://github.com/Muath477/frast1.git", "main"   # the user's own fork (never the original repository)
REPO = pathlib.Path("/content/frast1") if IN_COLAB else pathlib.Path(os.environ.get("ROOTIQ_REPO", "frast1")).resolve()
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
elif IN_COLAB:  # a rerun in the same Colab session: pick up knowledge-base / training-data changes (a local checkout is never touched)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
sys.path.insert(0, str(REPO / "backend"))
sys.path.insert(0, str(REPO / "training"))

from app.collectors import simulator as rq_sim
from app.intelligence.anomaly import FEATURES, vectors_from_recording
from app.intelligence.thresholds import static_severity
from app.intelligence.explain import template, grounded
from app.agents.playbooks import PLAYBOOKS
from app.knowledge import get_kb
import build_dataset as rq_data      # training/build_dataset.py  (vendor-knowledge data, deterministic)
import kb_eval                       # training/kb_eval.py        (offline scorer for the vendor-knowledge tests)
import external as rq_external       # training/external.py       (licence-gated Hugging Face data)



def _commit():
    try:
        return subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False).stdout.strip() or "unknown"
    except Exception:  # noqa: BLE001
        return "unknown"


_kb = get_kb()
assert _kb.validate() == [], "the knowledge base failed its own validation"
print("RootIQ code:", REPO, "| commit:", _commit(), "| features:", len(FEATURES), "| knowledge base:", _kb.stats()["vendors"], "vendors,", _kb.stats()["problems"], "problems")

## Stage A — Anomaly model (CPU)

**What it learns:** what a *healthy* lab looks like across six signals at once (link utilization/latency/loss, web latency, DNS latency, server CPU).
**How it is used by RootIQ:** the file `iforest.joblib` is loaded by `MultivariateScorer`; a score above 0.6 is added to the incident as *supporting evidence* (it can never decide the root cause).
**Honest expectation:** the static thresholds already catch these three scenarios quickly. The model earns its place by catching *combinations* that individually stay under the thresholds and by giving you a measured false-positive rate.
**Better data:** if you record the real lab (`RECORD_EVENTS=1` in RootIQ writes `data/recordings/*.jsonl`), put a healthy recording at `RootIQ_AI/data/healthy_recording.jsonl` and this stage trains on it instead of the simulator.

In [ ]:
#@title A1) Build the training and test data
import numpy as np


class _Sink:
    def __init__(self):
        self.events = []

    async def ingest(self, ev):
        self.events.append(ev)


LAG = 4  # seconds after an injection before a window counts as "anomalous"


def rollout(scenario=None, steps=900, inject_at=300, seed=0, every=2):
    """Simulate `steps` seconds of the lab; return one feature vector every `every` s and its timestamp."""
    async def go():
        random.seed(seed)
        sink, last, X, T = _Sink(), {}, [], []
        sim = rq_sim.Simulator(sink)
        for t in range(steps):
            if scenario and t == inject_at:
                sim.inject(scenario)
            await sim.step(1.0)
            for ev in sink.events:
                last[f"{ev.source_id}|{ev.metric}"] = ev.value
            sink.events.clear()
            if t % every == every - 1 and all(f in last for f in FEATURES):
                X.append([last[f] for f in FEATURES])
                T.append(t)
        return np.array(X), np.array(T)

    return run_sync(go())


REAL = ROOT / "data" / "healthy_recording.jsonl"
if REAL.exists():
    X_all = vectors_from_recording(str(REAL))
    cut = int(len(X_all) * 0.8)
    X_train, X_val, DATA_SOURCE = X_all[:cut], X_all[cut:], "real lab recording"
else:
    n_seeds = 3 if SMOKE else 8
    X_train = np.vstack([rollout(seed=s)[0] for s in range(n_seeds)])
    X_val = np.vstack([rollout(seed=500 + s)[0] for s in range(2)])
    DATA_SOURCE = "RootIQ simulator"

tests = {}
for sc in rq_sim.SCENARIOS:
    tests[sc] = [rollout(sc, steps=600, inject_at=200, seed=1000 + k) for k in range(2 if SMOKE else 4)]
print(f"{DATA_SOURCE}: train {X_train.shape}, validation {X_val.shape}, scenarios {list(tests)}")

In [ ]:
#@title A2) Train, measure and save the Isolation Forest
import joblib, sklearn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

model = IsolationForest(n_estimators=200, contamination=0.01, random_state=7).fit(X_train)


def prod_score(X):
    """Exactly the mapping RootIQ uses at runtime (app/intelligence/anomaly.py)."""
    return np.clip(-model.decision_function(np.atleast_2d(X)) * 4 + 0.5, 0, 1)


def static_hit(X):
    """The product's current detector: any feature over its warning threshold."""
    out = []
    for row in np.atleast_2d(X):
        out.append(any(static_severity(f.split("|")[1], v) >= 0.5 for f, v in zip(FEATURES, row)))
    return np.array(out)


THR = 0.6
s_val = prod_score(X_val)
thr_1pct = float(np.quantile(s_val, 0.99))  # a threshold that would give ~1% false alarms on healthy validation data
report = {"scenarios": {}}
neg_pre, pos_scores = [], []
for sc, runs in tests.items():
    tp_if, tp_1pct, tp_static, delays_if, delays_static, n_pos = 0, 0, 0, [], [], 0
    for X, T in runs:
        pre, post = T < 200, T >= 200 + LAG
        neg_pre.append(prod_score(X[pre]))
        sp = prod_score(X[post])
        pos_scores.append(sp)
        n_pos += post.sum()
        tp_if += (sp >= THR).sum()
        tp_1pct += (sp >= thr_1pct).sum()
        tp_static += static_hit(X[post]).sum()
        hit = np.where(prod_score(X) >= THR)[0]
        hit = [int(T[i]) - 200 for i in hit if T[i] >= 200]
        delays_if.append(min(hit) if hit else None)
        hit_s = [int(T[i]) - 200 for i, r in enumerate(static_hit(X)) if r and T[i] >= 200]
        delays_static.append(min(hit_s) if hit_s else None)
    report["scenarios"][sc] = {
        "tpr_iforest@0.6": round(float(tp_if / n_pos), 3),
        "tpr_iforest@1%fpr": round(float(tp_1pct / n_pos), 3),
        "tpr_static": round(float(tp_static / n_pos), 3),
        "delay_s_iforest": delays_if,
        "delay_s_static": delays_static,
    }
neg = np.concatenate([s_val] + neg_pre)
pos = np.concatenate(pos_scores)
report.update({
    "fpr_iforest@0.6": round(float((neg >= THR).mean()), 4),
    "fpr_static": round(float(np.concatenate([static_hit(X_val)] + [static_hit(X[T < 200]) for runs in tests.values() for X, T in runs]).mean()), 4),
    "auc": round(float(roc_auc_score(np.r_[np.zeros(len(neg)), np.ones(len(pos))], np.r_[neg, pos])), 4),
    "threshold_for_1pct_fpr": round(thr_1pct, 3),
})
print(json.dumps(report, indent=2))

meta = {
    "trained_at": now(), "data_source": DATA_SOURCE, "n_train": int(len(X_train)), "features": FEATURES,
    "sklearn": sklearn.__version__, "numpy": np.__version__, "threshold": THR, "metrics": report,
    "note": "Copy iforest.joblib to <repo>/data/models/iforest.joblib. Retrain with the sklearn version of backend/requirements.txt if loading warns.",
}
joblib.dump(model, ROOT / "models" / "iforest.joblib")
save_json(ROOT / "models" / "iforest.meta.json", meta)
save_json(ROOT / "reports" / "anomaly_report.json", meta)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3.2))
    plt.hist(neg, bins=40, alpha=.7, label="healthy")
    plt.hist(pos, bins=40, alpha=.7, label="fault")
    plt.axvline(THR, color="k", ls="--", label="RootIQ threshold 0.6")
    plt.legend(); plt.xlabel("anomaly score"); plt.tight_layout()
    plt.savefig(ROOT / "reports" / "anomaly_scores.png", dpi=120)
    print("saved plot")
except Exception as e:  # noqa: BLE001
    print("plot skipped:", e)
print("Saved model + report to", ROOT)

## Stage B — Data for the language model

There is very little good public networking data, so most of it is **made here and verified automatically**:

* **Vendor knowledge (B0, from the repo):** 43 vendors, their OS families, versions, command tables, syslog formats and 25 problem patterns turned into 9 tasks (identify a device, normalize syslog, find the right command, translate commands between vendors, diagnose a problem, how a change is saved / committed / rolled back per vendor, ...) in English and Arabic. Every label is computed from the knowledge base, not written by a model. It also teaches the model to **say "no curated command" where the KB has none** and to **refuse to run/approve** anything.
* **RootIQ-grounded tasks** (free, exact): incident facts → explanation (EN/AR) and "what should I do?" answers. The assistant answer must pass RootIQ's own `grounded()` check (no number that is not in the facts).
* **Teacher paraphrases** (Groq): a bigger model rewrites some answers for style variety — kept only if it still passes the same check.
* **Networking tutor** (Groq): ~55 CCNA-level topics, each explained in English and Arabic.
* **Public Hugging Face data (B4, optional, off by default):** only datasets whose licence the catalog allows (`zilalzihar/mikrotik-routeros-qa-dataset`, `witfoo/syslog-to-artifact`), screened for secrets/length/duplicates, **train only**.
* **Held out for evaluation only:** `Elfsong/Cisco_CCNA` (343 exam-style questions) and the repo's `test_seen` / `test_unseen` vendor files. They are never used for training.
* **Not used:** `ndavidson/cisco_inam_chatml` (33k rows derived from Cisco product documentation, licence not stated), the Junos CLI datasets (no licence stated) — fine for experiments, risky for something you ship. See `training/catalog/CATALOG.md`.

The Groq teacher is `openai/gpt-oss-120b` (open weights, Apache-2.0). Free-tier limits are low, so calls retry on HTTP 429 and every answer is cached on Drive: rerunning never repeats a paid/limited call. **B0 and B4 need no Groq key.**

In [ ]:
#@title B0) Vendor knowledge data from the repo (43 vendors, 25 problems, 10 tasks, EN + AR — no API calls)
from collections import Counter

KB_DIR = REPO / "training" / "data" / "generated"
_manifest = json.loads((KB_DIR / "manifest.json").read_text(encoding="utf-8"))
if _manifest["kb_fingerprint"] != rq_data.kb_fingerprint():
    print("The committed files are older than the knowledge base -> rebuilding in memory (commit the regenerated files afterwards).")
    kb_splits, _ = rq_data.build()
else:
    kb_splits = {s: kb_eval.load_rows(KB_DIR / f"{s}.jsonl") for s in rq_data.SPLITS}


def per_task(rows, n, seed=5):
    """Stratified cap: at most n rows per task (used for SMOKE and for the slower generation-based evaluation)."""
    rnd, by = random.Random(seed), {}
    for x in rows:
        by.setdefault(x["task"], []).append(x)
    out = []
    for t in sorted(by):
        items = by[t][:]
        rnd.shuffle(items)
        out += items[:n]
    return out


if SMOKE:
    kb_splits = {s: per_task(v, 12 if s == "train" else 4) for s, v in kb_splits.items()}
print("vendor-KB rows:", {s: len(v) for s, v in kb_splits.items()})
print("train tasks:", dict(Counter(r["task"] for r in kb_splits["train"])))

In [ ]:
#@title B1) Groq teacher with retry + Drive cache
import httpx

GROQ_KEY = secret("GROQ_API_KEY")
TEACHER = "openai/gpt-oss-120b"   # verify the name with GET https://api.groq.com/openai/v1/models if a call returns 404
CACHE = ROOT / "cache" / "teacher.jsonl"
_cache = {}
if CACHE.exists():
    for line in CACHE.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        _cache[r["k"]] = r["v"]


def teacher(system: str, user: str, max_tokens: int = 900, temperature: float = 0.4):
    key = hashlib.sha256(json.dumps([TEACHER, system, user, max_tokens, temperature], ensure_ascii=False).encode()).hexdigest()
    if key in _cache:
        return _cache[key]
    for attempt in range(8):
        try:
            r = httpx.post(
                "https://api.groq.com/openai/v1/chat/completions",
                headers={"authorization": f"Bearer {GROQ_KEY}"},
                json={"model": TEACHER, "max_tokens": max_tokens, "temperature": temperature,
                      "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]},
                timeout=90,
            )
        except httpx.HTTPError:
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 429:
            time.sleep(min(float(r.headers.get("retry-after", 15)), 90))
            continue
        if r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        text = (r.json()["choices"][0]["message"].get("content") or "").strip()
        if text:
            _cache[key] = text
            with CACHE.open("a", encoding="utf-8") as f:
                f.write(json.dumps({"k": key, "v": text}, ensure_ascii=False) + "\n")
        return text or None
    return None


print("Teacher check:", teacher("Answer in five words.", "What does OSPF stand for?", max_tokens=200))

In [ ]:
#@title B2) Networking tutor data (Groq teacher, EN + AR)
SYLLABUS = [
    "OSI and TCP/IP layers", "Ethernet frames and MAC addresses", "VLANs and 802.1Q trunking", "Spanning Tree (STP/RSTP)",
    "EtherChannel and LACP", "IPv4 addressing and subnetting", "VLSM and route summarization", "IPv6 addressing", "ARP",
    "ICMP, ping and traceroute", "DHCP", "DNS resolution", "NAT and PAT", "Static routing", "OSPF single-area basics",
    "OSPF neighbor states", "EIGRP basics", "BGP basics", "HSRP and VRRP", "Access control lists", "QoS: classification and marking",
    "Policing versus shaping", "Queueing and congestion", "SNMP", "Syslog and severity levels", "NetFlow and telemetry",
    "NTP", "SSH and AAA", "Port security", "DHCP snooping and Dynamic ARP Inspection", "Wireless LAN basics", "CDP and LLDP",
    "MTU and fragmentation", "TCP three-way handshake", "TCP congestion control", "UDP and when to use it", "HTTP and TLS",
    "IPsec VPN basics", "GRE tunnels", "Cloud networking: VPC and security groups", "SD-WAN concepts",
    "Network automation with Netmiko", "NETCONF, RESTCONF and YANG", "Troubleshooting methodology (bottom-up, divide and conquer)",
    "Interface errors: CRC, duplex mismatch", "Diagnosing link congestion and packet loss", "Latency and jitter",
    "Capacity planning", "Monitoring and alert fatigue", "Root cause analysis and event correlation", "Configuration backup and change management",
    "High availability and redundancy", "Load balancing basics", "Network segmentation and zero trust", "DNS failure troubleshooting",
]
TUTOR_SYSTEM = ("You are a precise network engineering instructor (CCNA level). Explain what it is, why it matters, how it works, "
                "a short Cisco IOS example only when relevant, and one common mistake. Maximum 170 words. "
                "If you are unsure of a detail, say so instead of inventing it.")
LANG_NOTE = {"en": "Answer in English.",
             "ar": "Answer in clear Modern Standard Arabic; keep protocol names, device names and commands in English."}
AR_CHARS = re.compile(r"[؀-ۿ]")
REFUSAL = re.compile(r"(?i)(as an ai|i cannot|i can't help|لا أستطيع|لا يمكنني)")


def good_tutor_answer(text, lang):
    if not text or not (200 <= len(text) <= 1800) or REFUSAL.search(text):
        return False
    if lang == "ar":
        return len(AR_CHARS.findall(text)) / max(len(text), 1) > 0.3
    return len(AR_CHARS.findall(text)) == 0


topics = SYLLABUS[:4] if SMOKE else SYLLABUS
tutor = []
for topic in topics:
    for lang in ("en", "ar"):
        q = f"Explain: {topic}. {LANG_NOTE[lang]}"
        a = teacher(TUTOR_SYSTEM, q)
        if good_tutor_answer(a, lang):
            tutor.append({"topic": topic, "lang": lang, "messages": [
                {"role": "system", "content": "You are a network engineering tutor. Be accurate and say when you are unsure."},
                {"role": "user", "content": f"{topic}" if lang == "en" else f"اشرح: {topic}"},
                {"role": "assistant", "content": a}]})
print(f"tutor examples kept: {len(tutor)} / {len(topics) * 2}")

In [ ]:
#@title B3) RootIQ-grounded tasks: explanation, recommendation (facts in -> checked answer out)
RQ_SYSTEM = ("You are RootIQ Copilot, a read-only network operations assistant. Use only the FACTS given. "
             "Never invent numbers, devices or commands, and keep the confidence percentage from the facts. "
             "You cannot approve or execute anything: changes need a named human engineer. "
             "Answer in the user's language; keep device names and commands in English.")
LINKS = ["R1 Gi0/0 → SW1 Gi0/1", "R1 Gi0/1 → SW2 Gi0/1", "SW1 Gi0/3 → APP01 eth0"]
rng = random.Random(7)


def make_facts(kind):
    conf = rng.randint(58, 99)
    n = rng.choice([1, 2, 3])
    if kind == "link":
        lat0 = rng.randint(1, 6)
        return {"label": rng.choice(LINKS), "conf": conf, "util": rng.randint(86, 100), "speed": rng.choice([10, 100, 1000]),
                "lat0": lat0, "lat": lat0 + rng.randint(40, 120), "lead": rng.randint(2, 12), "n": n}
    if kind == "svc-dns":
        return {"label": "Internal DNS", "conf": conf, "dns": rng.randint(0, 72), "dnsLat": rng.randint(300, 1200), "n": n}
    return {"label": "APP-01", "conf": conf, "cpu": rng.randint(90, 100), "http": rng.randint(600, 1800), "n": n}


KIND_TO_PB = {"link": "link", "svc-dns": "svc-dns", "server": "server"}


def recommend_answer(kind, facts, lang):
    pb = PLAYBOOKS[KIND_TO_PB[kind]]
    steps = "; ".join(f"{s['n']}) {s['title'].format(label=facts['label'])}" for s in pb["steps"])
    rb = "; ".join(r["title"] for r in pb["rollback"])
    if lang == "en":
        return (f"Recommended action: {pb['description']} Steps: {steps}. Rollback: {rb}. "
                f"Risk: {pb['risk']}. This needs a named engineer's approval; I cannot execute it.")
    return (f"الإجراء المقترح: {pb['description']} الخطوات: {steps}. التراجع: {rb}. المخاطرة: {pb['risk']}. "
            "يحتاج هذا موافقة مهندس بالاسم ولا أستطيع تنفيذه.")


def grounded_record(kind, facts, task, lang):
    lang_name = {"en": "English", "ar": "Arabic"}[lang]
    if task == "explain":
        user = f"FACTS: {json.dumps(facts, ensure_ascii=False)}\nTASK: Explain the incident in {lang_name} in 2-3 sentences."
        answer, check_against = template(kind, facts)[lang], facts
    else:
        pb = PLAYBOOKS[KIND_TO_PB[kind]]
        ctx = {**facts, "playbook": {k: pb[k] for k in ("description", "steps", "rollback", "risk")}}
        user = f"FACTS: {json.dumps(ctx, ensure_ascii=False)}\nTASK: What should the engineer do? Answer in {lang_name}."
        answer, check_against = recommend_answer(kind, facts, lang), ctx
    ok = grounded(answer, check_against)
    return ok, {"kind": kind, "task": task, "lang": lang, "facts": check_against, "messages": [
        {"role": "system", "content": RQ_SYSTEM}, {"role": "user", "content": user}, {"role": "assistant", "content": answer}]}


n_per_kind = 25 if SMOKE else 220
grounded_set, rejected = [], 0
for kind in ("link", "svc-dns", "server"):
    for _ in range(n_per_kind):
        f = make_facts(kind)
        for task in ("explain", "recommend"):
            for lang in ("en", "ar"):
                ok, rec = grounded_record(kind, f, task, lang)
                if ok:
                    grounded_set.append(rec)
                else:
                    rejected += 1
print(f"template-grounded examples: {len(grounded_set)} (rejected by the grounding check: {rejected})")

# Teacher paraphrases for style variety: kept only if they pass the SAME grounding check and keep a headline percentage.
PARA_SYSTEM = ("Rewrite the incident explanation for a network operations engineer in 2 short sentences. Use ONLY the numbers in FACTS. "
               "Keep the confidence percentage. Do not add advice. ")
paraphrased, para_seen = [], 0
for rec in grounded_set:
    if rec["task"] != "explain" or para_seen >= (6 if SMOKE else 150):
        continue
    para_seen += 1
    lang_name = {"en": "English", "ar": "Arabic (keep device names in English)"}[rec["lang"]]
    txt = teacher(PARA_SYSTEM + f"Write it in {lang_name}.", f"FACTS: {json.dumps(rec['facts'], ensure_ascii=False)}\nDRAFT: {rec['messages'][2]['content']}", max_tokens=400, temperature=0.7)
    if txt and grounded(txt, rec["facts"]) and any(p in txt for p in re.findall(r"\d+%", rec["messages"][2]["content"])) and not REFUSAL.search(txt):
        paraphrased.append({**rec, "messages": [rec["messages"][0], rec["messages"][1], {"role": "assistant", "content": txt}]})
print(f"teacher paraphrases kept: {len(paraphrased)} / {para_seen}")

In [ ]:
#@title B4) Optional: public Hugging Face data (only datasets the catalog approves for training; train-only)
EXT_LIMIT = {"zilalzihar/mikrotik-routeros-qa-dataset": 1500, "witfoo/syslog-to-artifact": 500}   # witfoo is an auxiliary mix: keep it small
external_rows = []
if INCLUDE_EXTERNAL:
    _cat = json.loads((REPO / "training" / "catalog" / "hf_catalog.json").read_text(encoding="utf-8"))["datasets"]
    for name in rq_external.CONVERTERS:
        if not rq_external.allowed(name):
            print("skipped (not approved for training in the catalog):", name)
            continue
        got = rq_external.load_external(name, limit=20 if SMOKE else EXT_LIMIT[name])
        print(f"{name}: {len(got)} rows | licence {_cat[name]['license']} | catalog use: {_cat[name]['use']}")
        external_rows += got
else:
    print("INCLUDE_EXTERNAL = False -> only the repo's own data + the tutor/grounded data is used (recommended for the first run).")
print("external rows:", len(external_rows))

In [ ]:
#@title B5) Assemble, split and save (train / validation / held-out tests)
rng2 = random.Random(11)
rng2.shuffle(grounded_set)
held_out = [r for r in grounded_set if r["task"] == "explain"][: (12 if SMOKE else 60)]
held_ids = {id(r) for r in held_out}
pool = [r for r in grounded_set if id(r) not in held_ids] + paraphrased + tutor
rng2.shuffle(pool)
cut = max(1, int(len(pool) * 0.05))
val, train = pool[:cut], pool[cut:]

# Vendor-knowledge rows keep the repo's own splits (the two test files are never trained on); external rows are train-only.
train += kb_splits["train"] + external_rows
val += kb_splits["val"]
rng2.shuffle(train)


def dump(path, rows):
    with pathlib.Path(path).open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


dump(ROOT / "data" / "train.jsonl", train)
dump(ROOT / "data" / "val.jsonl", val)
dump(ROOT / "data" / "eval_grounded.jsonl", held_out)
dump(ROOT / "data" / "kb_test_seen.jsonl", kb_splits["test_seen"])
dump(ROOT / "data" / "kb_test_unseen.jsonl", kb_splits["test_unseen"])
print(f"train {len(train)} | val {len(val)} | held-out grounding test {len(held_out)} | KB tests seen {len(kb_splits['test_seen'])} / unseen {len(kb_splits['test_unseen'])} -> {ROOT / 'data'}")
print("train mix by task:", dict(Counter(r.get("task", "?") for r in train).most_common()))

## Stage C — Fine-tune (QLoRA)  ·  needs a GPU runtime

Runtime → Change runtime type → GPU (T4 works for the 4B model; L4/A100 recommended for ALLaM-7B).

| Base model | Why | Rough GPU memory (4-bit) |
|---|---|---|
| `Qwen/Qwen3-4B-Instruct-2507` (default) | Strong multilingual small model, Apache-2.0, fast to tune | ~5 GB |
| `humain-ai/ALLaM-7B-Instruct-preview` | Saudi-built Arabic/English model, Apache-2.0 (same family Groq serves as `allam-2-7b`) | ~8 GB (slower) |

**What to expect (honest):** fine-tuning will not change the RCA (that is deterministic code). It can make the tutor answers more consistent, keep numbers grounded and give you an offline model. The evaluation cells tell you whether it actually helped; if not, keep using the base model or Groq.

Before training, cell C1 measures the **base** model on the CCNA benchmark, on the held-out grounding test and on the **vendor-knowledge tests** (`test_seen`: facts it was trained on in new wording; `test_unseen`: KB items it never saw — does it invent commands or say it does not know?) so the comparison is fair.

In [ ]:
#@title C0) Evaluation helpers: CCNA multiple-choice accuracy + grounding checks (used before and after training)
from datasets import load_dataset

ccna = []
for split in ("volume1", "volume2"):
    for r in load_dataset("Elfsong/Cisco_CCNA", split=split):
        ccna.append({"q": r["Question"], "choices": r["Choices"], "answer": "".join(sorted(re.findall(r"[A-G]", r["Answer"].upper())))})
random.Random(3).shuffle(ccna)
CCNA_N = 40 if SMOKE else len(ccna)
ccna = ccna[:CCNA_N]
MCQ_SYSTEM = "You are a CCNA exam expert. Reply with ONLY the letter(s) of the correct answer(s), for example B or DF."


def parse_letters(text):
    """'B', 'DF', 'Answer: D', 'A and D' -> sorted unique letters A-G (first line only)."""
    line = (text or "").strip().splitlines()[0] if (text or "").strip() else ""
    tokens = re.findall(r"\b[A-G]{1,6}\b", line.upper())
    multi = [t for t in tokens if len(t) > 1]
    return "".join(sorted(set(multi[0] if multi else "".join(tokens))))


def mcq_accuracy(answer_fn, items):
    ok = single_ok = single_n = 0
    for it in items:
        pred = parse_letters(answer_fn([{"role": "system", "content": MCQ_SYSTEM}, {"role": "user", "content": f"{it['q']}\n{it['choices']}"}]))
        hit = pred == it["answer"]
        ok += hit
        if len(it["answer"]) == 1:
            single_n += 1
            single_ok += hit
    return {"n": len(items), "accuracy": round(ok / len(items), 4), "single_answer_accuracy": round(single_ok / max(single_n, 1), 4)}


def grounding_eval(answer_fn, rows):
    ok = pct = lang_ok = refusals = 0
    for r in rows:
        text = answer_fn(r["messages"][:2]) or ""
        ok += bool(grounded(text, r["facts"]))
        pcts = re.findall(r"\d+%", r["messages"][2]["content"])
        pct += (not pcts) or any(p in text for p in pcts)
        is_ar = len(AR_CHARS.findall(text)) / max(len(text), 1) > 0.2
        lang_ok += (is_ar == (r["lang"] == "ar"))
        refusals += bool(REFUSAL.search(text))
    n = len(rows)
    return {"n": n, "grounded_rate": round(ok / n, 4), "keeps_headline_percent": round(pct / n, 4), "correct_language": round(lang_ok / n, 4), "refusals": refusals}


def teacher_answer(messages):
    return teacher(messages[0]["content"], messages[1]["content"], max_tokens=700, temperature=0.0)


base_teacher = mcq_accuracy(teacher_answer, ccna[: (15 if SMOKE else 80)])
print("Groq teacher on CCNA (reference ceiling):", base_teacher)

In [ ]:
#@title C0b) Vendor-knowledge evaluation set (scored offline by training/kb_eval.py: exact JSON, verbatim commands, invented/unsafe commands)
kb_seen_rows = per_task(kb_splits["test_seen"], 4 if SMOKE else 12, seed=21)
kb_unseen_rows = per_task(kb_splits["test_unseen"], 4 if SMOKE else 12, seed=22)
print(f"KB evaluation rows: {len(kb_seen_rows)} seen / {len(kb_unseen_rows)} unseen "
      f"(tasks with fewer than 5 rows are flagged low_n in the report)")

In [ ]:
#@title C1) Load the base model in 4-bit and measure it BEFORE training
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "Switch the runtime to GPU (Runtime -> Change runtime type)."
BASE = {"qwen": "Qwen/Qwen3-4B-Instruct-2507", "allam": "humain-ai/ALLaM-7B-Instruct-preview"}[BASE_KEY]
MODEL_NAME = "rootiq-network-v1"
use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| base:", BASE, "| bf16:", use_bf16)

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")


@torch.no_grad()
def generate(m, messages, max_new_tokens=160):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(m.device)
    out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


@torch.no_grad()
def generate_batch(m, conversations, max_new_tokens=256, batch_size=8):
    """Greedy decoding for many prompts at once (left padding). Used for the vendor-knowledge tests."""
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(conversations), batch_size):
        texts = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in conversations[i:i + batch_size]]
        enc = tok(texts, return_tensors="pt", padding=True).to(m.device)
        gen = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += [tok.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for g in gen]
    return outs


def kb_evaluate(m):
    """Vendor-knowledge scores for test_seen and test_unseen (see training/kb_eval.py for what each number means)."""
    res = {}
    for name, rows in (("seen", kb_seen_rows), ("unseen", kb_unseen_rows)):
        answers = generate_batch(m, [r["messages"][:2] for r in rows])
        res[name] = kb_eval.evaluate_answers(rows, answers)
    return res


held_out_rows = [json.loads(l) for l in (ROOT / "data" / "eval_grounded.jsonl").read_text(encoding="utf-8").splitlines()]
model.eval()
before = {
    "ccna": mcq_accuracy(lambda m_: generate(model, m_, 8), ccna),
    "grounding": grounding_eval(lambda m_: generate(model, m_, 160), held_out_rows),
    "kb": kb_evaluate(model),
}
print("BEFORE training:", json.dumps({k: v for k, v in before.items() if k != "kb"}, indent=2))
print("BEFORE training, vendor knowledge:", json.dumps({n: kb_eval.headline(r) for n, r in before["kb"].items()}, indent=2))
save_json(ROOT / "reports" / "eval_before.json", {"base": BASE, "measured_at": now(), **before, "groq_teacher_ccna": base_teacher})

In [ ]:
#@title C2) Train with QLoRA (checkpoints go to Drive; rerun this cell to resume after a disconnect)
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

train_rows = [json.loads(l) for l in (ROOT / "data" / "train.jsonl").read_text(encoding="utf-8").splitlines()]
val_rows = [json.loads(l) for l in (ROOT / "data" / "val.jsonl").read_text(encoding="utf-8").splitlines()]
train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_rows])
val_ds = Dataset.from_list([{"messages": r["messages"]} for r in val_rows])

# A smoke run and the full run must never share checkpoints: resuming the full run from the 20-step smoke checkpoint
# would inherit a finished learning-rate schedule.
RUN_NAME = MODEL_NAME + ("-smoke" if SMOKE else "")
CKPT = ROOT / "checkpoints" / RUN_NAME
tok.padding_side = "right"   # generate_batch (C1) switches it to left for decoding; training needs right padding
common = dict(
    output_dir=str(CKPT), per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=8, learning_rate=2e-4,
    lr_scheduler_type="cosine", warmup_ratio=0.03, logging_steps=5, save_steps=50, save_total_limit=2,
    eval_strategy="steps", eval_steps=50, bf16=use_bf16, fp16=not use_bf16, gradient_checkpointing=True,
    report_to="none", packing=False,
)
common.update(max_steps=20) if SMOKE else common.update(num_train_epochs=2)
try:
    cfg = SFTConfig(max_length=1024, **common)
except TypeError:  # older TRL
    cfg = SFTConfig(max_seq_length=1024, **common)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
try:
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tok, peft_config=lora)
except TypeError:  # older TRL
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tok, peft_config=lora)

last = sorted(CKPT.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
trainer.train(resume_from_checkpoint=str(last[-1]) if last else None)
ADAPTER = ROOT / "models" / f"{RUN_NAME}-lora"
trainer.model.save_pretrained(str(ADAPTER))
tok.save_pretrained(str(ADAPTER))
print("LoRA adapter saved to", ADAPTER)

## Stage D — Did it help? (ship / don't-ship)

The same CCNA questions, the same held-out incident facts and the same vendor-knowledge tests are answered again by the tuned model. The decision rule is deliberately strict:

* CCNA accuracy must **not drop** (more than 1 point) versus the base model, and
* the grounding rate on held-out incidents must be **at least 98%** and the answer must keep the headline percentage, and
* on the vendor-knowledge tests: **zero unsafe commands** (a change command that is not one of the KB's approved fix commands), **no regression** on identify / syslog / command lookup / problem diagnosis / configuration model (how a change is saved or committed) versus the base model, **`invented_command_rate` on unseen items not worse than the base model**, and the floors below for the two JSON tasks.

The floors (`KB_FLOOR`, 0.80) are **starting values**, not measured ones: after your first real run, look at the numbers and set them deliberately.

If it fails, do not connect it to RootIQ.

In [ ]:
#@title D1) Evaluate the tuned model, write the report, print the decision
tuned = trainer.model
tuned.eval()
tuned.config.use_cache = True
after = {
    "ccna": mcq_accuracy(lambda m_: generate(tuned, m_, 8), ccna),
    "grounding": grounding_eval(lambda m_: generate(tuned, m_, 160), held_out_rows),
    "kb": kb_evaluate(tuned),
}
delta = round(after["ccna"]["accuracy"] - before["ccna"]["accuracy"], 4)

KB_FLOOR = {"identify_exact": 0.80, "syslog_exact": 0.80}   # starting values: tune after the first real run
hb, ha = kb_eval.headline(before["kb"]["seen"]), kb_eval.headline(after["kb"]["seen"])
kb_rules = {
    "no_regression_vs_base": all((ha[k] or 0) >= (hb[k] or 0) for k in ("identify_exact", "syslog_exact", "command_lookup", "problem_diagnose", "config_model")),
    "floors_met": all((ha[k] or 0) >= v for k, v in KB_FLOOR.items()),
    "zero_unsafe_commands": after["kb"]["seen"]["unsafe_command_count"] == 0 and after["kb"]["unseen"]["unsafe_command_count"] == 0,
    "not_more_inventive_on_unseen": after["kb"]["unseen"]["invented_command_rate"] <= before["kb"]["unseen"]["invented_command_rate"] + 0.02,
}
ship = (delta >= -0.01 and after["grounding"]["grounded_rate"] >= 0.98 and after["grounding"]["keeps_headline_percent"] >= 0.95
        and all(kb_rules.values()))
report = {"base": BASE, "model_name": MODEL_NAME, "measured_at": now(), "smoke": SMOKE, "before": before, "after": after,
          "ccna_delta": delta, "kb_rules": kb_rules, "groq_teacher_ccna": base_teacher, "decision": "SHIP" if ship else "DO NOT SHIP"}
save_json(ROOT / "reports" / "eval_report.json", report)
print(json.dumps({k: v for k, v in report.items() if k not in ("before", "after")}, indent=2))
print("\nVendor knowledge, BEFORE -> AFTER (test_seen):")
for k in hb:
    print(f"  {k:24s} {hb[k]} -> {ha[k]}")
print("Vendor knowledge on items the model never saw (test_unseen):", json.dumps(kb_eval.headline(after["kb"]["unseen"])))
print("\n=== DECISION:", report["decision"], "===")
if SMOKE:
    print("(SMOKE run: numbers only prove the pipeline works. Set SMOKE=False and rerun B-D for the real result.)")

## Stage E — Save the model and connect it to RootIQ

RootIQ talks to any **OpenAI-compatible** server through `LLM_PROVIDER=custom`. Pick where the merged model runs:

| Where | How | Good for |
|---|---|---|
| **Your laptop (Ollama)** | convert the merged model to GGUF (llama.cpp `convert_hf_to_gguf.py`), `ollama create rootiq-network -f Modelfile`, then use `http://localhost:11434/v1` | the real hackathon demo (no internet, no quota) if the laptop can run it |
| **Colab (vLLM + tunnel)** | `pip install vllm`, `vllm serve <merged-dir> --served-model-name rootiq-network`, expose port 8000 with `cloudflared tunnel --url http://localhost:8000` | rehearsals only — the session ends when Colab disconnects |
| **Hugging Face Inference Endpoint** | push the merged model to a private repo and deploy it | a stable URL for judges, paid |

Then start RootIQ with:

```
LLM_ENABLED=1
LLM_PROVIDER=custom
LLM_BASE_URL=http://localhost:11434/v1     # or your tunnel / endpoint URL + /v1
LLM_MODEL=rootiq-network
CUSTOM_LLM_API_KEY=                          # only if your server needs one
```

The Copilot and the explanation agent keep their guards (numbers must be grounded, refusals rejected, deterministic fallback), so a weaker model can lower quality but cannot make RootIQ act.

In [ ]:
#@title E1) Merge the LoRA into the base model and save the full model to Drive
from peft import AutoPeftModelForCausalLM

assert report["decision"] == "SHIP" or SMOKE, "Evaluation says DO NOT SHIP: do not export this model."
del trainer, tuned, model
torch.cuda.empty_cache()
merged = AutoPeftModelForCausalLM.from_pretrained(str(ADAPTER), torch_dtype=dtype, device_map="auto").merge_and_unload()
OUT = ROOT / "models" / "merged" / RUN_NAME
merged.save_pretrained(str(OUT), safe_serialization=True)
tok.save_pretrained(str(OUT))
save_json(OUT / "rootiq_model_card.json", {"base": BASE, "name": MODEL_NAME, "trained_at": now(), "decision": report["decision"],
                                              "ccna_accuracy": report["after"]["ccna"], "grounding": report["after"]["grounding"],
                                              "vendor_kb": {n: kb_eval.headline(r) for n, r in report["after"]["kb"].items()}, "kb_rules": report["kb_rules"]})
print("Merged model saved to", OUT)

## Improved system prompt for a "Master Network AI" (advisor)

Your draft prompt is a good start. What changes make it work in practice:

* A prompt cannot *connect* to Drive, Hugging Face or Kaggle — code does. State what tools the model really has, and that everything else is unknown.
* Keep orchestration in code (RootIQ's Orchestrator) and make the Master AI an **advisor with read-only tools**, so a hallucination can never execute a change.
* Ask for structured output so other agents (and tests) can consume it.

```
You are the Network Advisor for RootIQ, a network operations platform. Audience: NOC engineers and the platform's software agents.

You can use ONLY the tools and context provided in this conversation (incident facts, topology, playbooks, retrieved documents, the vendor knowledge base).
If the needed information is not in them, say what is missing and how to get it. Never invent devices, numbers, commands or sources.

Rules
1. You advise. You never approve, execute or change anything. Every proposed command is text for a named human engineer to review.
2. Explain the why and the how at CCNA level, briefly; add one common mistake when it is useful.
3. Cite the context you used as [1], [2]. Keep numbers exactly as given.
4. If confidence is low or evidence conflicts, say "needs investigation" and list the next 2-3 checks.
5. Reply in the user's language (Arabic or English); keep device names and commands in English.
6. Vendor knowledge: give commands only for vendors/OS families present in the context, marked read-only; if the vendor, OS or command is not covered, say "not covered" and point to the vendor documentation. Never translate a command from memory.

Output (JSON when called by another agent, plain text for people):
{"answer": "...", "confidence": "high|medium|low", "sources": [1, 2], "needs_human": true|false, "next_checks": ["..."]}
```